In [0]:
from pyspark.sql import functions as F

failures = []

def check(condition, message):
    if condition:
        print(f"PASS: {message}")
    else:
        failures.append(message)
        print(f"FAIL: {message}")

# Bronze sanity checks 
comtrade_bronze = spark.table("afcfta_trade.bronze.comtrade_raw")
wits_bronze = spark.table("afcfta_trade.bronze.wits_tariffs_raw")

check(comtrade_bronze.count() == 357, f"comtrade_raw has 357 rows (found {comtrade_bronze.count()})")
check(wits_bronze.count() == 24, f"wits_tariffs_raw has 24 rows (found {wits_bronze.count()})")

# Silver checks
trade_flows = spark.table("afcfta_trade.silver.trade_flows")
tariffs = spark.table("afcfta_trade.silver.tariffs")

check(
    trade_flows.filter(F.col("reporter_iso3").isNull()).count() == 0,
    "silver.trade_flows has no null reporter_iso3"
)
check(
    trade_flows.filter(F.col("hs_chapter").isNull()).count() == 0,
    "silver.trade_flows has no null hs_chapter"
)
check(
    trade_flows.select("reporter_iso3").distinct().count() == 5,
    "silver.trade_flows covers all 5 countries"
)
check(
    tariffs.select("reporter_iso3").distinct().count() == 4,
    "silver.tariffs covers exactly 4 countries (Egypt excluded, documented)"
)

# Gold checks 
gold = spark.table("afcfta_trade.gold.market_opportunity")

check(gold.count() == 60, f"gold.market_opportunity has 60 rows (found {gold.count()})")
check(
    gold.filter(F.col("hs_chapter").isNull() | F.col("reporter_iso3").isNull()).count() == 0,
    "gold.market_opportunity has no null HS chapter or reporter"
)
check(
    gold.filter(
        (F.col("applied_tariff_pct") < 0) | (F.col("applied_tariff_pct") > 100) |
        (F.col("mfn_tariff_pct") < 0) | (F.col("mfn_tariff_pct") > 100)
    ).count() == 0,
    "all tariff percentages fall within 0-100 range"
)
check(
    gold.filter(F.col("total_trade_value_usd") < 0).count() == 0,
    "no negative trade values"
)

#  Final result 
if failures:
    raise Exception(f"{len(failures)} test(s) failed: {failures}")
else:
    print(f"\nAll tests passed.")